In [ ]:
%matplotlib inline
import os
os.environ['PY3_PROD'] = '1'

%load_ext autoreload
%autoreload 2
os.system('kinit')

In [ ]:
import numpy as np
import math
import pandas as pd
from typing import Union, List
import datetime
from wtpy.wrapper import WtDataHelper
from wtpy import WtDtServo
from pycmqlib3.utility.process_wt_data import *

In [ ]:
config_folder='C:/dev/wtdev'
dtServo = WtDtServo()
dtServo.setBasefiles(folder=f"{config_folder}/common/")
dtServo.setStorage(path=f'{config_folder}/storage/')

In [ ]:
from wtpy.wrapper import WtDataHelper
cont = "CF0905"
period = "day"
exch = "CZCE"
out_folder = r"c:/dev/wtdev/storage/his"
filename = '%s/%s/%s/%s.dsb' % (out_folder, period, exch, cont)

dtHelper = WtDataHelper()
curr_df = dtHelper.read_dsb_bars(filename, isDay=True)
curr_df.to_df()

In [ ]:
# this is wrong processing (read from dtservo and write by dthelper)

In [ ]:
# contmths_1 = ['501', '502', '503', '504', '505', '506', '507', '2509', '2510', '2511', '2512', '2601', '2602', '2603', '2604', '2605', '2606', '2607', '2608']

# products = ["MA", "CY", "FG", "CF", "SR", "ZC", "PX", "SR", "RM", "PR", "SH", "TA", "PL", 
#             "SA", "PF", "OI", "AP", "CJ", "UR", "RS", "SM", "SF", "WH", "PK", "PM", "RI", "LR", "JR"]
# periods = ['day', 'min1', 'min5']

# exch = 'CZCE'
# src_folder = 'c:/dev/wtdev/storage/his'
# for product in products:
#     for cont_mth in contmths:
#         cont = f"{product}{cont_mth}"
#         if exch == 'CZCE':
#             cont_key = f"{exch}.{product}.2{cont_mth}"
#         else:
#             cont_key = f"{exch}.{product}.{cont_mth}"
#         for period in periods:
#             if period == 'day':
#                 p_key = 'd'
#             else:
#                 p_key = period[0] + period[-1]
            
#             bar_df = dtServo.get_bars(cont_key, period=p_key, fromTime=202101010000, endTime=202508221520)
#             if bar_df is not None:
#                 bar_df = bar_df.to_df()
#                 bar_df = bar_df.rename(columns={"money": "turnover", "hold": "open_interest", 'bartime': 'time'})
#                 bar_df['time'] = bar_df['time'] - 199000000000
            
#                 file_folder = '%s/%s/%s' % (src_folder, period, exch)
#                 #print(cont, file_folder, p_key, bar_df.tail(5))
#                 res = save_bars_to_dsb(bar_df.copy(), cont, folder_loc=file_folder, period =p_key)
#                 print(product, cont, period, res)

In [ ]:
import os
import re
import shutil
from wtpy.wrapper import WtDataHelper

# If you don't already have these in scope:
dtHelper = WtDataHelper()

def period_to_pkey(period: str) -> str:
    """Map folder period name to save_bars_to_dsb period key."""
    return 'd' if period == 'day' else (period[0] + period[-1])  # "min1"->"m1", "min5"->"m5"

def copy_to_out(infile: str, out_folder: str, period: str, exch: str, new_contract: str) -> str:
    """
    Create out dir {out_folder}/{period}/{exch}, and copy infile to {new_contract}.dsb.
    Returns destination path.
    """
    outdir = os.path.join(out_folder, period, exch)
    os.makedirs(outdir, exist_ok=True)
    outfile = os.path.join(outdir, f"{new_contract}.dsb")
    shutil.copy2(infile, outfile)
    print(f"[COPY] {os.path.basename(infile)} -> {new_contract}.dsb")
    return outfile

def process_czce_folder(src_folder: str,
                        out_folder: str,
                        periods = ('day', 'min1', 'min5'),
                        exch: str = 'CZCE'):
    """
    Walk {src_folder}/{period}/{exch}, and:
      - For 4-digit YYMM:
          * 2509–2608: drop leading '2' (becomes 3-digit) then handle as split range
          * <=1412: copy unchanged (same name) to out folder
      - For 3-digit YMM:
          * <=412  -> copy as XX2YMM
          * >=609  -> copy as XX1YMM
          * 501–608 -> READ, preprocess, SPLIT at 2020-01-01, SAVE as XX1YMM (pre-2020) and XX2YMM (>=2020)
    """
    for period in periods:
        p_key = period_to_pkey(period)
        is_day = (period == 'day')
        in_dir = os.path.join(src_folder, period, exch)
        if not os.path.isdir(in_dir):
            print(f"[WARN] Missing folder: {in_dir}")
            continue

        for fname in os.listdir(in_dir):
            if not fname.endswith('.dsb'):
                continue

            infile = os.path.join(in_dir, fname)
            contract = fname[:-4]  # strip .dsb
            m = re.match(r'^([A-Za-z]{2})(\d{3,4})$', contract)
            if not m:
                print(f"[SKIP] Unexpected name format: {fname}")
                continue

            prefix, digits = m.groups()

            # ----- 4-digit YYMM handling -----
            if len(digits) == 4:
                d4 = int(digits)
                if 2509 <= d4 <= 2608:
                    # Normalize by removing the leading '2' (e.g., 2509 -> 509)
                    digits = digits[1:]
                    # fall through to 3-digit logic below
                elif d4 <= 1412:
                    # Copy unchanged (same contract name into out folder)
                    copy_to_out(infile, out_folder, period, exch, contract)
                    continue
                else:
                    print(f"[WARN] 4-digit out-of-scope: {contract}")
                    continue

            # ----- 3-digit YMM handling (original or normalized) -----
            d3 = int(digits)

            # Case A: <= 412 -> XX2YMM (copy)
            if d3 <= 412:
                new_contract = f"{prefix}2{digits}"
                copy_to_out(infile, out_folder, period, exch, new_contract)
                continue

            # Case B: >= 609 -> XX1YMM (copy)
            if d3 >= 609:
                new_contract = f"{prefix}1{digits}"
                copy_to_out(infile, out_folder, period, exch, new_contract)
                continue

            # Case C: 501–608 -> split by date 2020-01-01
            if 501 <= d3 <= 608:
                # Read the .dsb
                print(infile)
                bar_df = dtHelper.read_dsb_bars(infile, isDay=is_day).to_df()

                # --- preprocessing before save ---
                # rename columns and adjust time
                rename_map = {"money": "turnover", "hold": "open_interest", "bartime": "time"}
                cols_to_rename = {k: v for k, v in rename_map.items() if k in bar_df.columns}
                if cols_to_rename:
                    bar_df = bar_df.rename(columns=cols_to_rename)
                if "time" in bar_df.columns:
                    if is_day:
                        bar_df["time"] = 0
                    else:
                        bar_df["time"] = bar_df["time"] - 199000000000

                # ensure integer date
                if "date" in bar_df.columns:
                    bar_df["date"] = bar_df["date"].astype("int64")
                else:
                    print(f"[WARN] No 'date' column in {contract}, skipping split.")
                    continue

                cutoff = 20200101
                pre_df  = bar_df[bar_df["date"] <  cutoff]
                post_df = bar_df[bar_df["date"] >= cutoff]

                # Build out folder once
                outdir = os.path.join(out_folder, period, exch)
                os.makedirs(outdir, exist_ok=True)

                # Save each side using your wrapper (creates {outdir}/{contract}.dsb)
                pre_contract  = f"{prefix}1{digits}"   # pre-2020 -> 201x
                post_contract = f"{prefix}2{digits}"   # 2020+  -> 202x

                # Map period string to p_key for saving
                # (We already computed p_key above)
                if not pre_df.empty:
                    save_bars_to_dsb(pre_df, pre_contract,  folder_loc=outdir, period=p_key)
                    print(f"[SPLIT] {contract} pre-2020 -> {pre_contract}.dsb ({len(pre_df)})")
                else:
                    print(f"[SPLIT] {contract} pre-2020 -> (no rows)")

                if not post_df.empty:
                    save_bars_to_dsb(post_df, post_contract, folder_loc=outdir, period=p_key)
                    print(f"[SPLIT] {contract} 2020+    -> {post_contract}.dsb ({len(post_df)})")
                else:
                    print(f"[SPLIT] {contract} 2020+    -> (no rows)")

                continue

            print(f"[WARN] 3-digit out-of-scope: {contract}")

In [ ]:
src_folder = r"c:/dev/wtdev/storage/his"
out_folder = r"c:/dev/data/his_out"
periods = ['day', 'min1', 'min5']
process_czce_folder(src_folder, out_folder, periods, exch='CZCE')

In [ ]:
import os
import glob
import re
def rename_files(start_dir):
    for dirpath, dirnames, filenames in os.walk(start_dir):
        for fname in filenames:
            if not fname.endswith('.dsb'):
                continue

            infile = os.path.join(dirpath, fname)
            contract = fname[:-4]  # strip .dsb
            m = re.match(r'^([A-Z]{2})(\d{3})$', contract)
            if not m:
                print(f"[SKIP] Unexpected name format: {fname}")
                continue

            prefix, digits = m.groups()
            new_fname = f"{prefix}2{digits}.dsb"    
            new_path = os.path.join(dirpath, new_fname)
            print(f"Renaming '{infile}' to '{new_path}'")            
            try:
                # Perform the actual rename operation.
                os.rename(infile, new_path)
            except OSError as e:
                print(f"Error: Could not rename file {infile}. Reason: {e}")


In [ ]:
folder_to_process = "c:/dev/wtdev/storage/his/ticks/CZCE" 

rename_files(folder_to_process)

In [ ]:
import os
import json
import re
from typing import Any, Dict, List

# -------- Core conversion logic --------

_3DIGIT_RE = re.compile(r'^([A-Z]+)(\d{3})$')

def convert_czce_3digit_by_date(contract: str, date_int: int) -> str:
    """
    Convert a CZCE 3-digit contract (like CF609, CF009) into a 4-digit YYMM
    using the event date (YYYYMMDD). The resulting YYMM will be the smallest
    possible that is >= the date's YYMM (avoids mapping into the past).

    Examples:
      date=20160401, CF609 -> CF1609
      date=20260401, CF609 -> CF2609
      date=20200401, CF009 -> CF2009
      date=20300401, CF009 -> CF3009
      date=20191101, CF001 -> CF2001  (not CF1001, since 1001 < 1911)
    """
    m = _3DIGIT_RE.match(contract or "")
    if not m or not date_int:
        return contract  # leave non-3digit or missing-date untouched

    symbol, ymm3 = m.groups()
    y_digit = int(ymm3[0])          # single-digit "year within decade"
    mm = int(ymm3[1:])              # month 01..12

    # Derive YYMM for the date
    year_full = date_int // 10000
    month = (date_int // 100) % 100
    date_y = year_full % 100
    date_yymm = date_y * 100 + month

    # Start at the date's decade (e.g., 2019 -> decade YY=10), then search forward
    base_decade = (date_y // 10) * 10  # 10, 20, 30, ...
    for k in range(0, 10):  # search up to 10 decades ahead (should never need that many)
        YY = base_decade + y_digit + 10 * k
        cand_yymm = YY * 100 + mm
        if cand_yymm >= date_yymm:
            return f"{symbol}{YY:02d}{mm:02d}"

    # Fallback (theoretical only)
    YY = base_decade + y_digit
    return f"{symbol}{YY:02d}{mm:02d}"


# -------- JSON processing (only touches CZCE) --------

def _fix_event_obj(obj: Dict[str, Any]) -> Dict[str, Any]:
    """Fix a single event object with keys like 'date', 'from', 'to'."""
    date_val = obj.get("date", 0)
    for key in ("from", "to"):
        val = obj.get(key)
        if isinstance(val, str) and _3DIGIT_RE.match(val):
            obj[key] = convert_czce_3digit_by_date(val, date_val)
    return obj

def _process_czce_node(node: Any) -> Any:
    """
    Recursively process the CZCE subtree. We expect either:
      - dict of symbols -> list[events]
      - list[events]
      - nested dicts/lists (be tolerant)
    """
    if isinstance(node, list):
        out: List[Any] = []
        for item in node:
            if isinstance(item, dict):
                # Treat dict as an event; also recurse into nested structures if present
                fixed = _fix_event_obj(item.copy())
                for k, v in list(fixed.items()):
                    if isinstance(v, (dict, list)):
                        fixed[k] = _process_czce_node(v)
                out.append(fixed)
            else:
                out.append(item)
        return out

    if isinstance(node, dict):
        out: Dict[str, Any] = {}
        for k, v in node.items():
            if isinstance(v, list):
                out[k] = _process_czce_node(v)
            elif isinstance(v, dict):
                out[k] = _process_czce_node(v)
            else:
                out[k] = v
        return out

    return node  # primitives unchanged

def process_file_only_czce(src_path: str, dst_path: str) -> None:
    """Load JSON, adjust only the 'CZCE' branch, and write out."""
    with open(src_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "CZCE" in data:
        data = data.copy()
        data["CZCE"] = _process_czce_node(data["CZCE"])

    with open(dst_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


# -------- Runner for hots.json & seconds.json --------

def convert_hot_and_seconds(src_folder: str, target_folder: str) -> None:
    """
    Read:
      - {src_folder}/hots.json
      - {src_folder}/seconds.json
    Write:
      - {target_folder}/hot1.json  (adjusted hots)
      - {target_folder}/hot2.json  (adjusted seconds)
    Only the 'CZCE' subtree is modified.
    """
    os.makedirs(target_folder, exist_ok=True)

    in1 = os.path.join(src_folder, "hots.json")
    in2 = os.path.join(src_folder, "seconds.json")
    out1 = os.path.join(target_folder, "hot1.json")
    out2 = os.path.join(target_folder, "hot2.json")

    process_file_only_czce(in1, out1)
    process_file_only_czce(in2, out2)

    print(f"Done. Wrote:\n  {out1}\n  {out2}")


In [ ]:
src_folder = r"C:/dev/wtdev/common"
target_folder = r"c:/dev/wtdev/config"

convert_hot_and_seconds(src_folder, target_folder)